# Assignment


## Brief

Write the Python codes for the following questions.


## Instructions

- Step 1: Run the followings cell to get connected to your MongoDB. Make sure your credential is in dotenv file.
- Step 2: Put your answer inside each function. Please do not construct your own function.
- Step 3: You can test your function under test section.
- Step 4. Run test my function to confirm if my code is working.


### Connections

In [1]:
import os

from dotenv import load_dotenv
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

In [2]:
# Load environment variables from .env file
load_dotenv()
MONGODB_URI = os.getenv("MONGODB_URI")
if not MONGODB_URI:
    raise ValueError(
        "❌ MONGODB_URI not found!\n"
        "Please create a .env file with your MongoDB credentials.\n"
        "See README.md for setup instructions."
    )
client = MongoClient(MONGODB_URI, server_api=ServerApi("1"))
# Send a ping to confirm a successful connection
try:
    client.admin.command("ping")
    print("✅ Successfully connected to MongoDB!")
except Exception as e:
    print(e)

✅ Successfully connected to MongoDB!


In [3]:
db = client.sample_mflix
movies = db.movies
print(f"📊 Database: {db.name}")
print(f"📁 Collection: movies ({movies.count_documents({})} documents)")

📊 Database: sample_mflix
📁 Collection: movies (21349 documents)



### Question 1

Question: From the `movies` collection, return the documents with the `plot` that starts with `"war"` in acending order of released date, print only title, plot and released fields. Limit the result to 5.


**Answer:**

In [15]:
# ============================================================================
# Question 1
import pymongo
for m in movies.find({"plot": {"$regex": "^war", "$options": "i"}}, {"title": 1, "plot": 1, "released": 1}).sort('released', 1).limit(5):
    print(m)
# ============================================================================

{'_id': ObjectId('573a1398f29313caabce9508'), 'plot': 'Warrior/pacifist Princess Nausicaè desperately struggles to prevent two warring nations from destroying themselves and their dying planet.', 'title': 'Nausicaè of the Valley of the Wind', 'released': datetime.datetime(1984, 3, 11, 0, 0)}
{'_id': ObjectId('573a1398f29313caabce91ec'), 'plot': 'Warrior/pacifist Princess Nausicaè desperately struggles to prevent two warring nations from destroying themselves and their dying planet.', 'title': 'Nausicaè of the Valley of the Wind', 'released': datetime.datetime(1984, 3, 11, 0, 0)}
{'_id': ObjectId('573a1398f29313caabcebfc6'), 'plot': 'Warlords Kagetora and Takeda each wish to prevent the other from gaining hegemony in feudal Japan. The two samurai leaders pursue one another across the countryside, engaging in massive ...', 'title': 'Heaven and Earth', 'released': datetime.datetime(1991, 2, 8, 0, 0)}
{'_id': ObjectId('573a13b5f29313caabd44f06'), 'plot': 'Warning! This synopsis contains sp

### Question 2

Question: Group by `rated` and count the number of movies in each.

**Answer:**

In [16]:
# ============================================================================
# Question 2
stage_group_rated = {
   "$group": {
         "_id": "$rated",
         # Count the number of movies in the group:
         "movie_count": { "$sum": 1 }, 
   }
}

pipeline = [
   stage_group_rated,
]
results = movies.aggregate(pipeline)

# Loop through the 'rated-summary' documents:
for rated_summary in results:
   print(rated_summary)
# ============================================================================

{'_id': None, 'movie_count': 9894}
{'_id': 'PG', 'movie_count': 1852}
{'_id': 'G', 'movie_count': 477}
{'_id': 'TV-PG', 'movie_count': 76}
{'_id': 'TV-Y7', 'movie_count': 3}
{'_id': 'AO', 'movie_count': 3}
{'_id': 'OPEN', 'movie_count': 1}
{'_id': 'R', 'movie_count': 5537}
{'_id': 'PG-13', 'movie_count': 2321}
{'_id': 'APPROVED', 'movie_count': 709}
{'_id': 'TV-14', 'movie_count': 89}
{'_id': 'TV-MA', 'movie_count': 60}
{'_id': 'PASSED', 'movie_count': 181}
{'_id': 'Not Rated', 'movie_count': 1}
{'_id': 'TV-G', 'movie_count': 59}
{'_id': 'M', 'movie_count': 37}
{'_id': 'GP', 'movie_count': 44}
{'_id': 'Approved', 'movie_count': 5}



### Question 3

Question: Count the number of movies with 3 comments or more.


**Answer:**


In [18]:
# ============================================================================
# Question 3
stage_lookup_comments = {
   "$lookup": {
         "from": "comments", 
         "localField": "_id", 
         "foreignField": "movie_id", 
         "as": "related_comments",
   }
}
stage_limit_5 = { "$limit": 5 }
# Calculate the number of comments for each movie:
stage_add_comment_count = {
   "$addFields": {
         "comment_count": {
            "$size": "$related_comments"
         }
   } 
}

# Match movie documents with at least 1 comment:
stage_match_with_comments = {
   "$match": {
         "comment_count": {
            "$gte": 3
         }
   } 
}
pipeline = [
   stage_lookup_comments,
   stage_add_comment_count,
   stage_match_with_comments,
   stage_limit_5,
]

results = movies.aggregate(pipeline)
for movie in results:
   print(movie["title"])
   print("Comment count:", movie["comment_count"])

   for comment in movie["related_comments"][:5]:
         print(" * {name}: {text}".format(
            name=comment["name"],
            text=comment["text"]))
   print()

# ============================================================================

Upstream
Comment count: 3
 * Jordan Medina: Adipisci vel dolores tenetur sit inventore. Doloribus dolor nesciunt voluptas saepe veritatis. Mollitia eum iure ut nam.
 * Theresa Holmes: Unde ut eum doloremque expedita commodi exercitationem. Error soluta temporibus quasi. Libero quam nulla mollitia officia ipsa. Odio harum cupiditate a dignissimos.
 * Mace Tyrell: Assumenda quibusdam vel reprehenderit error. Optio voluptatibus maxime tempore velit. Architecto modi possimus officia minima eum quis quis.

The Wizard of Oz
Comment count: 139
 * Amy Phillips: Tempora ullam dignissimos tenetur possimus cum fuga amet laborum. Enim voluptatum voluptatem eum assumenda corrupti dolore quia quis. Ut eligendi provident neque.
 * Andrea Le: Odit corporis eveniet dicta itaque maiores fugit nihil. Similique vitae nulla consectetur esse consequatur consectetur. Doloribus nostrum labore vitae.
 * Andrea Le: Sint alias illum iusto deserunt beatae. Minima vel cupiditate porro voluptatem omnis veritatis. O